# Avaliação de imóveis
## Dados de mercado imobiliário

São imóveis ofertados em cidades de São Paulo, anúncios públicos na web. Para possibilitar uma avaliação de imóveis na região ponderada com dados socioeconômicos, é necessário coletar a informação do IBGE do último censo, 2022, para a menor região agregada (setor censitário).

Essa coleta atualmente é manual e demora em torno de 2 min por ponto (imóvel).

## Amostra de imóveis

In [2]:
import pandas as pd

# Carrega os dados do arquivo Excel para um DataFrame
# 'header=0' usa a primeira linha como nomes de colunas
# pandas infere automaticamente tipos para números e datas, e mantém o restante como string (object).
imoveis = pd.read_excel('amostra_imoveis.xlsx', header=0)

# Garante que a coluna 'COORDENADAS' seja tratada como string para o split
imoveis['COORDENADAS'] = imoveis['COORDENADAS'].astype(str)

# Quebra a coluna 'COORDENADAS' em duas, usando ',' como separador
# e cria novas colunas temporárias para LAT e LON.
coordinates_split = imoveis['COORDENADAS'].str.split(',', expand=True)

# Limpa e converte a Latitude (primeira parte)
# Remove quaisquer caracteres que não sejam dígitos, pontos ou hífens, e converte para float.
imoveis['LAT'] = coordinates_split[0].str.replace(r'[^-?\d.]', '', regex=True).astype(float)

# Limpa e converte a Longitude (segunda parte)
# Remove quaisquer caracteres que não sejam dígitos, pontos ou hífens, e converte para float.
imoveis['LON'] = coordinates_split[1].str.replace(r'[^-?\d.]', '', regex=True).astype(float)

# Opcionalmente, remove a coluna 'COORDENADAS' original, pois já foi processada.
imoveis = imoveis.drop(columns=['COORDENADAS'])

# Descarte das colunas especificadas pelo usuário
columns_to_drop = [
    'VU_AT', 'FRENTE', 'LADO', 'VIA', 'PAV',
    'TELEFONE/CÓDIGO', 'INFORMANTE', 'LINK', 'TÉCNICO'
]
imoveis = imoveis.drop(columns=columns_to_drop, errors='ignore')

print("DataFrame 'imoveis' carregado e processado com sucesso.")
print(imoveis.head())
print(imoveis.info())

DataFrame 'imoveis' carregado e processado com sucesso.
      TIPO        VT     AT      AC  VIA           DATA_COLETA MUNICÍPIO  \
0   Galpão  30000000  10200  8500.0     1  2026-03-09 00:00:00   Diadema   
1  Terreno  10000000   4060     NaN     1  2026-03-09 00:00:00   Diadema   
2  Terreno   8000000   5420   788.0     1  2026-03-09 00:00:00   Diadema   
3  Terreno  20000000   5000     NaN     0  2026-03-09 00:00:00   Diadema   
4  Terreno   6000000   7000     NaN     0  2026-03-09 00:00:00   Diadema   

           BAIRRO                              ENDEREÇO  \
0          Centro                Av. Luigi Papaiz, 843    
1         Taboão                    Rua Armando Pinelli   
2        Serraria   R. Álvares Cabral, 1210 - Conceição   
3   Vila Nogueira           R. Tibiriçá, 294 - Serraria   
4       Conceição               Rua Almirante Cockrane    

                   CARACTERIZAÇÃO        LAT        LON  
0                          Galpão -23.676139 -46.618044  
1        Terreno

## Georreferenciamento

In [3]:
import geopandas
from shapely.geometry import Point

# Cria a coluna de geometria a partir das colunas LAT e LON
imoveis['geometry'] = imoveis.apply(lambda row: Point(row['LON'], row['LAT']), axis=1)

# Converte o DataFrame para um GeoDataFrame
gdf_imoveis = geopandas.GeoDataFrame(imoveis, geometry='geometry')

# Define o Sistema de Referência de Coordenadas (CRS) como WGS84 (EPSG:4326)
gdf_imoveis.crs = 'EPSG:4326'

print('GeoDataFrame de imóveis criado com sucesso:')
print(gdf_imoveis.head())
print(gdf_imoveis.info())

GeoDataFrame de imóveis criado com sucesso:
      TIPO        VT     AT      AC  VIA           DATA_COLETA MUNICÍPIO  \
0   Galpão  30000000  10200  8500.0     1  2026-03-09 00:00:00   Diadema   
1  Terreno  10000000   4060     NaN     1  2026-03-09 00:00:00   Diadema   
2  Terreno   8000000   5420   788.0     1  2026-03-09 00:00:00   Diadema   
3  Terreno  20000000   5000     NaN     0  2026-03-09 00:00:00   Diadema   
4  Terreno   6000000   7000     NaN     0  2026-03-09 00:00:00   Diadema   

           BAIRRO                              ENDEREÇO  \
0          Centro                Av. Luigi Papaiz, 843    
1         Taboão                    Rua Armando Pinelli   
2        Serraria   R. Álvares Cabral, 1210 - Conceição   
3   Vila Nogueira           R. Tibiriçá, 294 - Serraria   
4       Conceição               Rua Almirante Cockrane    

                   CARACTERIZAÇÃO        LAT        LON  \
0                          Galpão -23.676139 -46.618044   
1        Terreno dentro da

Agora o DataFrame `imoveis` foi convertido para um GeoDataFrame (`gdf_imoveis`), com uma coluna `geometry` contendo os pontos georreferenciados. Isso o torna compatível para futuras operações espaciais, como a intersecção com polígonos de setores censitários do IBGE.

In [21]:
import folium

# Formata a coluna 'VT' para valor monetário em R$ sem decimais com separador de milhar
# Cria uma cópia para evitar SettingWithCopyWarning, se gdf_imoveis for uma view
gdf_imoveis_copy = gdf_imoveis.copy()
gdf_imoveis_copy['VT_FORMATADO'] = gdf_imoveis_copy['VT'].apply(lambda x: f'R$ {x:,.0f}'.replace(',', '.'))

# Cria o mapa interativo usando gdf_imoveis.explore()
# 'tiles' define a camada base do mapa (OpenStreetMap para visão de ruas)
# 'tooltip' e 'popup' podem exibir informações adicionais ao passar o mouse ou clicar no ponto
# 'marker_kwds' permite customizar os marcadores (ex: cor).

m = gdf_imoveis_copy.explore(
    tiles='OpenStreetMap',
    column='MUNICÍPIO', # Colorir os pontos por município
    tooltip=['ENDEREÇO', 'MUNICÍPIO', 'TIPO', 'VT_FORMATADO'], # Informações ao passar o mouse, usando a coluna formatada
    popup=True, # Popup ao clicar (mostrará todas as colunas)
    marker_kwds={'radius': 5, 'fill': True, 'fill_opacity': 0.7},
    cmap='viridis', # Colormap para diferenciar os municípios
    legend=True, # Exibir legenda
    style_kwds={'color': 'black', 'weight': 0.5}, # Estilo das bordas dos marcadores
    name='Imóveis'
)

# Para visualizar o mapa, ele precisa ser renderizado. Em ambiente Jupyter/Colab,
# basta colocar a variável do mapa na última linha da célula.

## Censo IBGE 2022

As informações censitárias são públicas, hospedadas no

In [30]:
import geopandas as gpd

# Define o caminho para o arquivo ZIP
zip_file_path = 'SP_setores_CD2022.zip'

# Constrói o URI para o shapefile dentro do ZIP
# Assume que o shapefile principal tem o mesmo nome base do arquivo ZIP com extensão .shp
# ATENÇÃO: Você pode precisar ajustar 'SP_setores_CD2022.shp' se o nome do arquivo SHP interno for diferente.
# Ex: se o shp interno for 'setores_sp.shp', a linha ficaria assim:
# shp_in_zip_uri = f"zip://{zip_file_path}!setores_sp.shp"
shp_in_zip_uri = f"zip://{zip_file_path}!SP_setores_CD2022.shp"

print(f"Tentando carregar o shapefile diretamente do ZIP: {shp_in_zip_uri}...")

# Carrega os setores censitários do IBGE diretamente do arquivo shapefile (.shp) dentro do ZIP
gdf_setores_censitarios = gpd.read_file(
    filename = shp_in_zip_uri
    # Não é necessário especificar 'layer' ou 'driver' explicitamente para um arquivo .shp simples
    # engine="pyogrio", # pyogrio pode não ser compatível com o esquema 'zip://' diretamente
    # use_arrow=True # use_arrow também pode não ser compatível
  )

# Verifica e, se necessário, reprojeta o CRS para WGS84 (EPSG:4326) para compatibilidade
# com o gdf_imoveis, caso o arquivo do IBGE não esteja neste CRS.
if gdf_setores_censitarios.crs != 'EPSG:4326':
    print(f"Reprojetando de {gdf_setores_censitarios.crs} para EPSG:4326...")
    gdf_setores_censitarios = gdf_setores_censitarios.to_crs('EPSG:4326')

print("GeoDataFrame de setores censitários do IBGE carregado com sucesso:")
print(gdf_setores_censitarios.head())
print(gdf_setores_censitarios.info())

Tentando carregar o shapefile diretamente do ZIP: zip://SP_setores_CD2022.zip!SP_setores_CD2022.shp...
Reprojetando de EPSG:4674 para EPSG:4326...
GeoDataFrame de setores censitários do IBGE carregado com sucesso:
          CD_SETOR SITUACAO CD_SIT CD_TIPO  AREA_KM2 CD_REGIAO NM_REGIAO  \
0  350010505000001   Urbana      1       0  0.123824         3   Sudeste   
1  350010505000002   Urbana      1       0  0.202614         3   Sudeste   
2  350010505000003   Urbana      1       0  0.235048         3   Sudeste   
3  350010505000004   Urbana      1       0  0.288468         3   Sudeste   
4  350010505000005   Urbana      1       0  0.213724         3   Sudeste   

  CD_UF      NM_UF   CD_MUN  ... NM_FCU CD_AGLOM NM_AGLOM CD_RGINT  \
0    35  São Paulo  3500105  ...   None     None     None     3505   
1    35  São Paulo  3500105  ...   None     None     None     3505   
2    35  São Paulo  3500105  ...   None     None     None     3505   
3    35  São Paulo  3500105  ...   None     None 

## Combinando geometrias ponto -> polígono

In [32]:
# Realiza a junção espacial (spatial join) dos imóveis com os setores censitários.
# O parâmetro 'predicate="within"' garante que estamos procurando por imóveis que estão 'dentro' de um setor.
# O 'how="left"' garante que todos os imóveis sejam mantidos, mesmo que não caiam em um setor (embora não esperado).
# Seleciona apenas a coluna 'CD_SETOR' do gdf_setores_censitarios para ser incorporada.
gdf_imoveis_com_setor = gpd.sjoin(gdf_imoveis, gdf_setores_censitarios[['CD_SETOR', 'geometry']], how='left', predicate='within')

print("Junção espacial concluída. GeoDataFrame 'gdf_imoveis_com_setor' criado com sucesso:")
display(gdf_imoveis_com_setor.head())
display(gdf_imoveis_com_setor.info())

Junção espacial concluída. GeoDataFrame 'gdf_imoveis_com_setor' criado com sucesso:


,TIPO,VT,AT,AC,VIA,DATA_COLETA,MUNICÍPIO,BAIRRO,ENDEREÇO,CARACTERIZAÇÃO,LAT,LON,geometry,index_right,CD_SETOR
0,Galpão,30000000,10200,8500.0,1,2026-03-09 00:00:00,Diadema,Centro,"Av. Luigi Papaiz, 843",Galpão,-23.676139,-46.618044,POINT (-46.61804 -23.67614),18182,351380105000570
1,Terreno,10000000,4060,NaN,1,2026-03-09 00:00:00,Diadema,Taboão,Rua Armando Pinelli,Terreno dentro da cidade,-23.667353,-46.612755,POINT (-46.61275 -23.66735),17857,351380105000098
2,Terreno,8000000,5420,788.0,1,2026-03-09 00:00:00,Diadema,Serraria,"R. Álvares Cabral, 1210 - Conceição",Terreno dentro da area urbano,-23.701295,-46.609709,POINT (-46.60971 -23.7013),18446,351380105000871
3,Terreno,20000000,5000,NaN,0,2026-03-09 00:00:00,Diadema,Vila Nogueira,"R. Tibiriçá, 294 - Serraria",Terreno dentro da area urbano,-23.700788,-46.609168,POINT (-46.60917 -23.70079),18301,351380105000697
4,Terreno,6000000,7000,NaN,0,2026-03-09 00:00:00,Diadema,Conceição,Rua Almirante Cockrane,Terreno com construções,-23.696303,-46.619607,POINT (-46.61961 -23.6963),18269,351380105000663


<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 38 entries, 0 to 37
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   TIPO            38 non-null     object  
 1   VT              38 non-null     int64   
 2   AT              38 non-null     int64   
 3   AC              6 non-null      float64 
 4   VIA             38 non-null     int64   
 5   DATA_COLETA     37 non-null     object  
 6   MUNICÍPIO       38 non-null     object  
 7   BAIRRO          38 non-null     object  
 8   ENDEREÇO        38 non-null     object  
 9   CARACTERIZAÇÃO  38 non-null     object  
 10  LAT             38 non-null     float64 
 11  LON             38 non-null     float64 
 12  geometry        38 non-null     geometry
 13  index_right     38 non-null     int64   
 14  CD_SETOR        38 non-null     object  
dtypes: float64(3), geometry(1), int64(4), object(7)
memory usage: 4.8+ KB


None

## Combina renda do setor censitário de outra base pública

In [34]:
# Carrega a base de dados de renda por setor censitário
df_renda = pd.read_excel('agregado_renda_br.xlsx')

# Seleciona as colunas chave do setor e a renda do responsável
# A coluna 'CD_SETOR' é a chave para a junção.
# 'V06004' é o valor do rendimento nominal médio mensal das pessoas responsáveis.
df_renda_selecionada = df_renda[['CD_SETOR', 'V06004']].copy()

# Renomeia a coluna V06004 para um nome mais descritivo
df_renda_selecionada = df_renda_selecionada.rename(columns={'V06004': 'REND_MEDIO_RESPONSAVEL'})

# Converte a coluna 'CD_SETOR' em df_renda_selecionada para string para evitar erros de tipo no merge
df_renda_selecionada['CD_SETOR'] = df_renda_selecionada['CD_SETOR'].astype(str)
# Converte a coluna 'CD_SETOR' em gdf_imoveis_com_setor para string para garantir compatibilidade
gdf_imoveis_com_setor['CD_SETOR'] = gdf_imoveis_com_setor['CD_SETOR'].astype(str)

print("DataFrame de renda carregado e colunas selecionadas/renomeadas com sucesso:")
display(df_renda_selecionada.head())

# Combina (merge) o GeoDataFrame de imóveis com o DataFrame de renda
# Usamos 'left merge' para garantir que todos os imóveis sejam mantidos.
# A junção é feita pela coluna 'CD_SETOR'.
gdf_imoveis_final = gdf_imoveis_com_setor.merge(
    df_renda_selecionada, on='CD_SETOR', how='left'
)

print("GeoDataFrame de imóveis combinado com os dados de renda por setor censitário:")
display(gdf_imoveis_final.head())
display(gdf_imoveis_final.info())

DataFrame de renda carregado e colunas selecionadas/renomeadas com sucesso:


,CD_SETOR,REND_MEDIO_RESPONSAVEL
0,110001505000002,2453.03
1,110001505000003,2126.44
2,110001505000004,1664.94
3,110001505000006,1722.4
4,110001505000007,1930.94


GeoDataFrame de imóveis combinado com os dados de renda por setor censitário:


,TIPO,VT,AT,AC,VIA,DATA_COLETA,MUNICÍPIO,BAIRRO,ENDEREÇO,CARACTERIZAÇÃO,LAT,LON,geometry,index_right,CD_SETOR,REND_MEDIO_RESPONSAVEL
0,Galpão,30000000,10200,8500.0,1,2026-03-09 00:00:00,Diadema,Centro,"Av. Luigi Papaiz, 843",Galpão,-23.676139,-46.618044,POINT (-46.61804 -23.67614),18182,351380105000570,2676.57
1,Terreno,10000000,4060,NaN,1,2026-03-09 00:00:00,Diadema,Taboão,Rua Armando Pinelli,Terreno dentro da cidade,-23.667353,-46.612755,POINT (-46.61275 -23.66735),17857,351380105000098,2899.19
2,Terreno,8000000,5420,788.0,1,2026-03-09 00:00:00,Diadema,Serraria,"R. Álvares Cabral, 1210 - Conceição",Terreno dentro da area urbano,-23.701295,-46.609709,POINT (-46.60971 -23.7013),18446,351380105000871,2091.2
3,Terreno,20000000,5000,NaN,0,2026-03-09 00:00:00,Diadema,Vila Nogueira,"R. Tibiriçá, 294 - Serraria",Terreno dentro da area urbano,-23.700788,-46.609168,POINT (-46.60917 -23.70079),18301,351380105000697,2867.33
4,Terreno,6000000,7000,NaN,0,2026-03-09 00:00:00,Diadema,Conceição,Rua Almirante Cockrane,Terreno com construções,-23.696303,-46.619607,POINT (-46.61961 -23.6963),18269,351380105000663,2767.84


<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   TIPO                    38 non-null     object  
 1   VT                      38 non-null     int64   
 2   AT                      38 non-null     int64   
 3   AC                      6 non-null      float64 
 4   VIA                     38 non-null     int64   
 5   DATA_COLETA             37 non-null     object  
 6   MUNICÍPIO               38 non-null     object  
 7   BAIRRO                  38 non-null     object  
 8   ENDEREÇO                38 non-null     object  
 9   CARACTERIZAÇÃO          38 non-null     object  
 10  LAT                     38 non-null     float64 
 11  LON                     38 non-null     float64 
 12  geometry                38 non-null     geometry
 13  index_right             38 non-null     int64   
 14  CD_SETOR            

None

## Plota o mapa inicial como heatmap de renda do responsável

In [37]:
import folium

# Formata a coluna 'VT' para valor monetário em R$ sem decimais com separador de milhar
# Cria a coluna VT_FORMATADO em gdf_imoveis_final
gdf_imoveis_final['VT_FORMATADO'] = gdf_imoveis_final['VT'].apply(lambda x: f'R$ {x:,.0f}'.replace(',', '.'))

# Converte a coluna 'REND_MEDIO_RESPONSAVEL' para numérica, forçando erros para NaN para não quebrar
gdf_imoveis_final['REND_MEDIO_RESPONSAVEL'] = pd.to_numeric(gdf_imoveis_final['REND_MEDIO_RESPONSAVEL'], errors='coerce')

# Define os bins para a legenda de renda
# Estes bins foram escolhidos com base nos valores típicos de renda média no Brasil, mas podem ser ajustados.
# Por exemplo, pode-se usar pd.qcut(gdf_imoveis_final['REND_MEDIO_RESPONSAVEL'], q=5) para bins baseados em quantis.
bins_renda = [0, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, gdf_imoveis_final['REND_MEDIO_RESPONSAVEL'].max() + 1]

# Cria o mapa interativo usando gdf_imoveis_final.explore()
# 'tiles' define a camada base do mapa (OpenStreetMap para visão de ruas)
# 'tooltip' e 'popup' podem exibir informações adicionais ao passar o mouse ou clicar no ponto
# 'marker_kwds' permite customizar os marcadores (ex: raio e opacidade).

m_renda = gdf_imoveis_final.explore(
    tiles='OpenStreetMap',
    column='REND_MEDIO_RESPONSAVEL', # Colorir os pontos pela renda média do responsável
    tooltip=['ENDEREÇO', 'MUNICÍPIO', 'TIPO', 'VT_FORMATADO', 'REND_MEDIO_RESPONSAVEL'], # Incluir a renda no tooltip
    popup=True, # Popup ao clicar
    marker_kwds={'radius': 5, 'fill': True, 'fill_opacity': 0.7},
    cmap='YlGnBu', # Colormap para diferenciar os níveis de renda (Amarelo-Verde-Azul é bom para dados sequenciais)
    legend=True, # Exibir legenda
    style_kwds={'color': 'black', 'weight': 0.5}, # Estilo das bordas dos marcadores
    name='Imóveis com Renda Média',
    bins=bins_renda # Define os bins para a legenda de faixas
)

# Para visualizar o mapa, ele precisa ser renderizado.
# Em ambiente Jupyter/Colab, basta colocar a variável do mapa na última linha da célula.
m_renda